# Kjemiske biblioteker

```{admonition} Læringsutbytte
Etter å ha arbeidet med dette temaet, skal du kunne:

1. hente og utforske grunnstoffdata med `mendeleev`
2. hente data om kjemiske forbindelser fra PubChem med `pubchempy`
3. representere molekyler fra SMILES og beregne molekylegenskaper med `RDKit`
4. balansere en reaksjonslikning ved å løse et lineært likningssystem
5. beregne pH i sammensatte løsninger med `pHcalc` og kontrollere resultatet med en egen numerisk metode
6. vurdere når et bibliotek er nyttig, og når det er mer hensiktsmessig å skrive koden selv
```

Python har et stort antall biblioteker, og flere av dem er utviklet spesielt for kjemi. I dette kapitlet skal vi se på noen biblioteker som gir tilgang til kjemiske data, molekylstrukturer og numeriske beregninger.

Når vi vurderer om et bibliotek er nyttig, kan vi spørre hva det tilfører programmet vårt. Biblioteker er særlig verdifulle når de gir tilgang til:

1. **Data:** kuraterte og kvalitetssikrede verdier, for eksempel ioniseringsenergier og atomradier
2. **Algoritmer og visualisering:** funksjonalitet som er omfattende eller krevende å programmere selv
3. **Standardisering:** filformater og konvensjoner som gjør det mulig å utveksle data mellom ulike programmer

For enkle faglige beregninger kan det derimot være mer lærerikt å skrive funksjonene selv. Da blir sammenhengen mellom kjemien og programkoden tydeligere, og koden kan lettere tilpasses nye problemstillinger.

Kapitlet er organisert etter hva bibliotekene brukes til:

| Del | Bruksområde | Eksempler |
|---|---|---|
| 1 | Kjemiske data | `mendeleev`, `pubchempy` |
| 2 | Struktur og representasjon | `RDKit` |
| 3 | Beregninger vi kan programmere selv | støkiometri og balansering |
| 4 | Numeriske beregninger | `pHcalc` |


## Installasjon

Før du kjører eksemplene, må bibliotekene installeres. Kjør kodecellen nedenfor én gang. Utropstegnet gjør at kommandoen sendes til pakkebehandleren `pip`, i stedet for å bli tolket som vanlig Python-kode.


In [ ]:
!pip install mendeleev pubchempy rdkit pHcalc sympy


## Biblioteker som gir tilgang til data

### Mendeleev

Periodesystemet kan behandles som et stort datasett. Egenskaper som elektronegativitet, ioniseringsenergi og atomradius er samlet fra målinger og modeller, og de kan ikke beregnes direkte med noen få linjer kode. Biblioteket `mendeleev` gir oss tilgang til slike grunnstoffdata.

Grunnenheten i biblioteket er klassen `element`. En klasse kan betraktes som en oppskrift for å opprette objekter. Alle grunnstoffobjektene har de samme typene egenskaper, men verdiene varierer fra grunnstoff til grunnstoff.


In [ ]:
from mendeleev import element

svovel = element("S")          # Eller: element(16)

print(svovel.name)
print(svovel.symbol)
print(svovel.atomic_number)
print(svovel.atomic_weight)
print(svovel.block, svovel.period, svovel.group_id)


Elektronegativitet hentes med et *metodekall*, ikke som en vanlig egenskap. Grunnen er at elektronegativitet kan defineres ved hjelp av flere skalaer. Vi må derfor angi hvilken skala vi vil bruke.


In [ ]:
print("Pauling: ", svovel.electronegativity("pauling"))
print("Allen:   ", svovel.electronegativity("allen"))
print("Mulliken:", svovel.electronegativity("mulliken"))


```{admonition} Underveisoppgave: Hvorfor er tallene så forskjellige?
:class: tip

De tre skalaene gir svært ulike tallverdier for svovel.

Slå opp definisjonene av skalaene. Hvilken enhet brukes i hvert tilfelle? Hvorfor kan tallene ikke sammenliknes direkte, og hva må vi gjøre før en sammenlikning blir meningsfull?
```

````{admonition} Løsningsforslag
:class: tip, dropdown

Paulings skala er enhetsløs og er definert ut fra bindingsenergier. Allens og Mullikens skalaer bygger derimot på energistørrelser og oppgis i elektronvolt (eV).

Tallverdiene kan derfor ikke sammenliknes direkte. Det som kan sammenliknes, er blant annet rekkefølgen grunnstoffene får på hver skala, eller relative forskjeller etter at skalaene er normert til samme intervall.

Dette illustrerer et generelt poeng: En tallverdi fra et bibliotek må alltid tolkes sammen med definisjonen og enheten som hører til.
````


#### Hele periodesystemet på én gang

Når vi skal studere trender, er det tungvint å opprette ett objekt for hvert grunnstoff. Funksjonen `fetch_table` henter i stedet hele tabellen i én operasjon og returnerer en **pandas-dataframe**. Da kan vi bruke metodene for datahåndtering som vi har arbeidet med tidligere.


In [ ]:
from mendeleev.fetch import fetch_table
import pandas as pd

grunnstoffer = fetch_table("elements")

kolonner = ["atomic_number", "symbol", "name", "period", "group_id",
            "block", "atomic_weight", "en_pauling", "covalent_radius_pyykko"]

grunnstoffer[kolonner].head(12)


#### Manglende verdier

Også kuraterte datasett kan inneholde manglende verdier. Noen størrelser er ikke definert for alle grunnstoffer, mens andre verdier ikke er målt eller beregnet.


In [ ]:
mangler = grunnstoffer[grunnstoffer["en_pauling"].isna()]
print("Antall grunnstoff uten Pauling-elektronegativitet:", len(mangler))
mangler[["atomic_number", "symbol", "name", "group_id"]]


```{admonition} Underveisoppgave: Hvilke verdier mangler?
:class: tip

Studer tabellen ovenfor.

1. Én gruppe i periodesystemet er tydelig representert blant de manglende verdiene. Hvilken gruppe er dette, og hva kan være den kjemiske forklaringen?
2. Flere av de øvrige grunnstoffene med manglende verdier har noe annet til felles. Hva?
3. Lag et plott av Pauling-elektronegativitet mot atomnummer for hele periodesystemet. Hvordan håndterer `matplotlib` de manglende verdiene?
```

````{admonition} Løsningsforslag
:class: tip, dropdown

1. **Edelgassene.** Paulings skala er definert med utgangspunkt i bindingsenergier. For grunnstoffer som i liten grad danner bindinger, finnes det derfor ikke alltid et godt grunnlag for å angi en verdi.
2. Flere av de tyngste grunnstoffene er fremstilt i svært små mengder og har korte halveringstider. Mange kjemiske egenskaper er derfor ikke bestemt eksperimentelt.
3. Manglende verdier blir representert som `NaN` og tegnes vanligvis ikke. Det kommer ikke nødvendigvis en feilmelding, og derfor bør vi alltid undersøke datasettet før vi tolker et plott.

```{code-block} python
import matplotlib.pyplot as plt

plt.figure(figsize=(9, 4))
plt.scatter(grunnstoffer["atomic_number"], grunnstoffer["en_pauling"], s=18)
plt.xlabel("Atomnummer")
plt.ylabel("Elektronegativitet (Pauling)")
plt.title("Elektronegativitet i periodesystemet")
plt.grid(alpha=0.3)
plt.show()
```
````


#### Trender innenfor en periode

Når dataene ligger i en dataframe, kan vi enkelt velge ut den delen av periodesystemet vi ønsker å studere.


In [ ]:
import matplotlib.pyplot as plt

periode2 = grunnstoffer[grunnstoffer["period"] == 2]

plt.figure(figsize=(7, 4))
plt.plot(periode2["atomic_number"], periode2["en_pauling"],
         marker="o", color="crimson")

for _, rad in periode2.iterrows():
    plt.annotate(rad["symbol"],
                 (rad["atomic_number"], rad["en_pauling"]),
                 textcoords="offset points", xytext=(0, 8), ha="center")

plt.xlabel("Atomnummer")
plt.ylabel("Elektronegativitet (Pauling)")
plt.title("Elektronegativitet i andre periode")
plt.grid(alpha=0.3)
plt.show()


```{admonition} Underveisoppgave: Forklar to trender
:class: tip

1. Plottet ovenfor viser andre periode (Z = 3 til 10). Beskriv trenden og forklar den ut fra kjerneladning og skjerming. Hvorfor mangler neon et punkt?
2. Lag tilsvarende plott for **gruppe 17** (halogenene) og for **gruppe 1**. Går trenden samme vei? Forklar.
3. Lag ett plott der du viser periode 2 og periode 3 i samme koordinatsystem, med hver sin farge og merkelapp. Hva sier sammenlikningen om effekten av å legge til et helt elektronskall?
```

````{admonition} Løsningsforslag
:class: tip, dropdown

1. Elektronegativiteten øker fra litium til fluor. Kjerneladningen øker med ett proton per steg, mens de nye elektronene går inn i det *samme* skallet og skjermer hverandre dårlig. Effektiv kjerneladning øker derfor, atomradien minker, og valenselektronene holdes hardere. Neon mangler verdi av samme grunn som de andre lette edelgassene: den danner ikke bindinger som kan brukes til å definere en Pauling-verdi.

2. I en gruppe går trenden **motsatt vei**: elektronegativiteten minker nedover. Valenselektronene havner i skall lenger fra kjernen og skjermes av flere fylte innerskall.

```{code-block} python
gruppe17 = grunnstoffer[grunnstoffer["group_id"] == 17]
gruppe1  = grunnstoffer[grunnstoffer["group_id"] == 1]

plt.plot(gruppe17["atomic_number"], gruppe17["en_pauling"],
         marker="o", label="Gruppe 17")
plt.plot(gruppe1["atomic_number"], gruppe1["en_pauling"],
         marker="s", label="Gruppe 1")
plt.xlabel("Atomnummer")
plt.ylabel("Elektronegativitet (Pauling)")
plt.legend()
plt.grid(alpha=0.3)
plt.show()
```

3. Kurven for periode 3 ligger systematisk lavere enn for periode 2, men har samme form. Formen skyldes økende effektiv kjerneladning innenfor perioden, mens nivåforskjellen skyldes det ekstra skallet.
````


`mendeleev` inneholder langt mer enn elektronegativitet: ioniseringsenergier, ioneradier, isotoper, oksidensjonstall og en del til. Du får en oversikt over alt som er registrert for ett grunnstoff ved å skrive objektet i en egen celle.


In [ ]:
# Ioniseringsenergier er en dictionary med ioniseringsgrad som nøkkel
natrium = element("Na")
print("1. ioniseringsenergi:", natrium.ionenergies[1], "eV")
print("2. ioniseringsenergi:", natrium.ionenergies[2], "eV")

# Skriv objektet alene i en celle for å se alt som finnes:
# natrium


```{admonition} Underveisoppgave: Det store spranget
:class: tip

Hent ut de fem første ioniseringsenergiene til magnesium og plott dem mot ioniseringsgrad.

Hvor kommer det store spranget, og hvorfor akkurat der? Hva forteller dette deg om elektronstrukturen til magnesium, og hvorfor magnesium danner Mg²⁺ og ikke Mg³⁺?
```

````{admonition} Løsningsforslag
:class: tip, dropdown

```{code-block} python
magnesium = element("Mg")

grader = [1, 2, 3, 4, 5]
energier = [magnesium.ionenergies[n] for n in grader]

plt.plot(grader, energier, marker="o")
plt.xlabel("Ioniseringsgrad")
plt.ylabel("Ioniseringsenergi (eV)")
plt.title("Ioniseringsenergier for magnesium")
plt.grid(alpha=0.3)
plt.show()
```

Spranget kommer mellom den andre og den tredje ioniseringa. De to første elektronene tas fra 3s-orbitalen i valensskallet. Det tredje må rives ut av det fylte 2p-skallet, som ligger mye nærmere kjernen og er mye hardere bundet.

Nettopp derfor stopper magnesium på Mg²⁺: energien som skal til for å ta det tredje elektronet, blir aldri betalt tilbake av gitterenergi eller hydratiseringsenergi i vanlig kjemi.
````


### PubChemPy

PubChem er en åpen database med informasjon om kjemiske forbindelser. Biblioteket `pubchempy` kommuniserer med PubChem gjennom et API og gjør dataene tilgjengelige som Python-objekter.

I motsetning til `mendeleev` henter `pubchempy` data over internett mens koden kjører. Det betyr at du må ha nettilgang, og at mange gjentatte oppslag kan ta tid. Når det er mulig, bør resultatene lagres lokalt og brukes på nytt.


In [ ]:
import pubchempy as pcp

treff = pcp.get_compounds("paracetamol", "name")
paracetamol = treff[0]

print("PubChem CID: ", paracetamol.cid)
print("Molekylformel:", paracetamol.molecular_formula)
print("Molar masse:  ", paracetamol.molecular_weight, "g/mol")
print("IUPAC-navn:   ", paracetamol.iupac_name)


```{admonition} Nettbaserte API-er kan endres
:class: warning

Nettbaserte tjenester og API-er kan endres over tid. Dersom et attributt eller et kodeeksempel slutter å virke, bør du kontrollere den oppdaterte dokumentasjonen for både `pubchempy` og PubChem.

Dette er en vanlig del av arbeid med programkode som henter data fra eksterne tjenester. Dokumentasjonen er derfor en viktig del av verktøyet.
```


In [ ]:
# CID-en er nøkkelen videre. Vi bruker den igjen i molekylvisualisering.
for navn in ["koffein", "aspirin", "ibuprofen", "askorbinsyre"]:
    forbindelse = pcp.get_compounds(navn, "name")[0]
    print(f"{navn:15} CID {forbindelse.cid:>8}   "
          f"{forbindelse.molecular_formula:>10}   "
          f"{forbindelse.molecular_weight} g/mol")


```{admonition} Underveisoppgave: Bygg din egen tabell
:class: tip

Lag ei liste med fem legemidler eller naturstoffer du er nysgjerrig på. Hent molekylformel, molar masse og CID for hvert av dem, og samle det i en pandas-dataframe.

Sorter tabellen etter molar masse og skriv den ut. Ta vare på CID-ene: du får bruk for dem når vi skal visualisere molekylene.
```

````{admonition} Løsningsforslag
:class: tip, dropdown

```{code-block} python
import pandas as pd

navn_liste = ["koffein", "nikotin", "morfin", "penicillin G", "kolesterol"]

rader = []
for navn in navn_liste:
    forbindelse = pcp.get_compounds(navn, "name")[0]
    rader.append({
        "navn": navn,
        "cid": forbindelse.cid,
        "formel": forbindelse.molecular_formula,
        "molar_masse": float(forbindelse.molecular_weight),
    })

tabell = pd.DataFrame(rader).sort_values("molar_masse")
tabell
```
````


## Biblioteker for struktur og representasjon

### RDKit

`RDKit` er et mye brukt bibliotek innen kjeminformatikk. Det arbeider med selve molekylstrukturen, ikke bare med molekylformelen. Biblioteket kan blant annet lese strukturkoder, identifisere bindinger og ringer, tegne strukturformler, søke etter strukturmønstre og generere tredimensjonale konformasjoner.

RDKit gjør det også mulig å bevege seg mellom ulike representasjoner av den samme forbindelsen:

- **Symbolsk:** SMILES-koden `CCO` representerer etanol.
- **Todimensjonal:** RDKit kan tegne en strukturformel.
- **Tredimensjonal:** biblioteket kan generere en konformasjon med bindingslengder og vinkler.
- **Beregnete egenskaper:** strukturen kan brukes til å beregne deskriptorer som molar masse, logP og polart overflateareal.

Representasjonene viser ulike sider av den samme forbindelsen.

#### SMILES

SMILES (Simplified Molecular Input Line Entry System) er en metode for å skrive en molekylstruktur som tekst. Noen grunnleggende regler er:

- Atomer skrives med grunnstoffsymbol. Hydrogenatomer utelates vanligvis og legges til automatisk.
- Tegn ved siden av hverandre angir bindinger: `CCO` representerer etanol.
- `=` angir dobbeltbinding, mens `#` angir trippelbinding.
- Parenteser brukes for sidegrupper: `CC(C)C` representerer 2-metylpropan.
- Tall brukes til å lukke ringer: `C1CCCCC1` representerer sykloheksan.
- Små bokstaver brukes for aromatiske atomer: `c1ccccc1` representerer benzen.


In [ ]:
from rdkit import Chem
from rdkit.Chem import Draw, Descriptors, AllChem
from rdkit.Chem.Draw import IPythonConsole   # gjør at molekyler tegnes automatisk

koffein = Chem.MolFromSmiles("CN1C=NC2=C1C(=O)N(C)C(=O)N2C")
koffein


```{admonition} Kontroller at innlesingen lyktes
:class: warning

`Chem.MolFromSmiles` returnerer `None` dersom SMILES-koden ikke er gyldig. Hvis denne verdien sendes videre til en annen funksjon, kan feilmeldingen oppstå et annet sted i programmet og bli vanskelig å tolke.

Det er derfor lurt å kontrollere resultatet:

```{code-block} python
molekyl = Chem.MolFromSmiles(smiles)
if molekyl is None:
    print("Ugyldig SMILES:", smiles)
```
```


#### Flere molekyler samtidig


In [ ]:
smiles = {
    "koffein":            "CN1C=NC2=C1C(=O)N(C)C(=O)N2C",
    "paracetamol":        "CC(=O)Nc1ccc(O)cc1",
    "acetylsalisylsyre":  "CC(=O)Oc1ccccc1C(=O)O",
    "ibuprofen":          "CC(C)Cc1ccc(cc1)C(C)C(=O)O",
}

molekyler = [Chem.MolFromSmiles(s) for s in smiles.values()]

Draw.MolsToGridImage(molekyler, legends=list(smiles.keys()),
                     molsPerRow=4, subImgSize=(240, 200))


#### Fra struktur til egenskap

Når RDKit har lest inn en struktur, kan biblioteket beregne en rekke molekyldeskriptorer. Noen beregnes direkte fra atomene og bindingene, mens andre er basert på empiriske modeller.


In [ ]:
import pandas as pd

rader = []
for navn, s in smiles.items():
    molekyl = Chem.MolFromSmiles(s)
    rader.append({
        "navn": navn,
        "molar_masse": round(Descriptors.MolWt(molekyl), 2),
        "logP": round(Descriptors.MolLogP(molekyl), 2),
        "TPSA": round(Descriptors.TPSA(molekyl), 1),
        "H-donorer": Descriptors.NumHDonors(molekyl),
        "H-akseptorer": Descriptors.NumHAcceptors(molekyl),
    })

pd.DataFrame(rader)


Legg merke til at kolonnene representerer ulike typer informasjon:

- **Molar masse** beregnes fra atomsammensetningen i strukturen og de atomvektene biblioteket bruker.
- **H-donorer** og **H-akseptorer** telles etter bestemte kjemiske definisjoner.
- **TPSA** (topologisk polart overflateareal) beregnes ved hjelp av fragmentbidrag fra polare atomer.
- **logP** er et modellestimat for fordelingen mellom oktanol og vann, basert på atomgruppene i molekylet.

En tabell fra et bibliotek kan altså inneholde både direkte strukturberegninger og modellbaserte estimater. Det er viktig å vite hvilken type verdi vi arbeider med før vi tolker resultatet.


```{admonition} Underveisoppgave: Lipinski
:class: tip

Lipinskis "regel om fem" er en tommelfingerregel for om en forbindelse har egenskaper som ligner på et legemiddel som kan tas som tablett. En forbindelse bryter regelen hvis mer enn ett av følgende er sant:

- molar masse over 500 g/mol
- logP over 5
- flere enn 5 H-bindingsdonorer
- flere enn 10 H-bindingsakseptorer

1. Skriv en funksjon `bryter_lipinski(smiles)` som returnerer antall brudd.
2. Test den på de fire forbindelsene ovenfor.
3. Test den så på kolesterol (`CC(C)CCCC(C)C1CCC2C1(CCC3C2CC=C4C3(CCC(C4)O)C)C`) og på et lite peptid du finner SMILES for. Hva ser du?
4. Regelen kalles en *tommelfingerregel*. Finn minst ett kjent legemiddel som bryter den. Hva sier det om hvor mye vekt du bør legge på slike regler?
```

````{admonition} Løsningsforslag
:class: tip, dropdown

```{code-block} python
def bryter_lipinski(smiles):
    molekyl = Chem.MolFromSmiles(smiles)
    if molekyl is None:
        raise ValueError("Ugyldig SMILES: " + smiles)

    brudd = 0
    if Descriptors.MolWt(molekyl) > 500:        brudd += 1
    if Descriptors.MolLogP(molekyl) > 5:        brudd += 1
    if Descriptors.NumHDonors(molekyl) > 5:     brudd += 1
    if Descriptors.NumHAcceptors(molekyl) > 10: brudd += 1
    return brudd

for navn, s in smiles.items():
    print(f"{navn:20} {bryter_lipinski(s)} brudd")
```

De fire små molekylene bryter ingen av kriteriene. Kolesterol bryter logP-kriteriet fordi forbindelsen er svært upolar. Et peptid vil ofte bryte flere kriterier samtidig.

Regelen har mange unntak, og enkelte kjente legemidler ligger utenfor ett eller flere av kriteriene. Den bør derfor brukes som en enkel vurderingsregel, ikke som en absolutt grense.
````


#### Finne funksjonelle grupper

Et bibliotek kan også brukes til å søke etter bestemte strukturmønstre i et molekyl. Mønstrene skrives med SMARTS, som bygger på SMILES, men i tillegg inneholder jokertegn og betingelser.


In [ ]:
monstre = {
    "karboksylsyre":  "[CX3](=O)[OX2H1]",
    "ester":          "[CX3](=O)[OX2][CX4]",
    "amid":           "[CX3](=O)[NX3]",
    "alkohol/fenol":  "[OX2H]",
    "aromatisk ring": "c1ccccc1",
}

rader = []
for navn, s in smiles.items():
    molekyl = Chem.MolFromSmiles(s)
    rad = {"navn": navn}
    for gruppe, smarts in monstre.items():
        mal = Chem.MolFromSmarts(smarts)
        rad[gruppe] = len(molekyl.GetSubstructMatches(mal))
    rader.append(rad)

pd.DataFrame(rader)


```{admonition} Underveisoppgave: Kontroller resultatet
:class: tip

Tabellen ovenfor er generert av et program. Bruk kjemikunnskapen din til å kontrollere resultatet.

1. Tegn acetylsalisylsyre for hånd og tell etter. Stemmer antallet ester- og karboksylsyregrupper?
2. Paracetamol har ifølge tabellen én amidgruppe. Ser du den i strukturformelen?
3. `[OX2H]`-mønsteret teller også OH-gruppa i en karboksylsyre. Er det riktig eller feil? Diskuter hvorfor et program ikke kan svare på det spørsmålet uten at du forteller det hva du er ute etter.
4. Legg til et mønster for **keton** og ett for **eter**. Test på de fire forbindelsene. Gikk det som du trodde?
```

````{admonition} Løsningsforslag
:class: tip, dropdown

1. Ja. Acetylsalisylsyre har én ester (acetylgruppa på fenol-oksygenet) og én karboksylsyre. Dette er en fin sjekk på at både du og programmet har rett.
2. Ja, `CC(=O)N`-delen. Paracetamol er et acetamid.
3. Begge deler, avhengig av hva du spør om. `[OX2H]` betyr "oksygen med to bindinger, hvorav ett hydrogen", og det er sant for karboksylsyrens OH. Vil du ha *bare* alkoholer, må du utelukke karbonylnaboen eksplisitt, for eksempel med `[OX2H][CX4]`. Programmet følger mønsteret du har definert. Det kan ikke avgjøre hvilken kjemisk avgrensning du egentlig ønsket.
4. Et generelt mønster for en karbonylgruppe kan også treffe estere, amider og karboksylsyrer. For å finne bare ketoner må begge nabogruppene til karbonylkarbonet avgrenses til karbonatomer. SMARTS-mønstre bør derfor alltid testes mot molekyler der du kjenner strukturen.
````


#### Fra 2D til 3D

En SMILES-kode beskriver hvilke atomer som er bundet sammen, men gir ikke én bestemt tredimensjonal geometri. RDKit kan generere en mulig 3D-struktur ved å legge til hydrogenatomer, lage en startgeometri og deretter optimere geometrien med et kraftfelt.

Resultatet er en **konformasjon**: én rimelig geometri som er beregnet med en klassisk modell. Den er ikke nødvendigvis den eneste eller den mest stabile konformasjonen molekylet kan ha.


In [ ]:
etanol = Chem.AddHs(Chem.MolFromSmiles("CCO"))
AllChem.EmbedMolecule(etanol, randomSeed=42)
AllChem.MMFFOptimizeMolecule(etanol)

molblokk = Chem.MolToMolBlock(etanol)
print(molblokk[:300])


Strengen `molblokk` bruker et standardisert filformat som andre kjemiprogrammer kan lese. I neste kapittel bruker vi den samme strengen til å tegne molekylet i tre dimensjoner. Dette viser hvorfor standardiserte formater er nyttige: ulike biblioteker kan utveksle strukturer uten å være utviklet sammen.


## Beregninger vi kan programmere selv

Flere biblioteker tilbyr ferdige funksjoner for enkle kjemiske beregninger, for eksempel stoffmengde, fortynning og cellepotensial. Slike funksjoner kan være praktiske, men i et introduksjonsemne er det ofte mer lærerikt å programmere de grunnleggende sammenhengene selv.

Det har to fordeler:

1. **Den faglige sammenhengen blir synlig.** En funksjon for stoffmengde bygger direkte på $n=m/M$. Når vi skriver funksjonen selv, ser vi hvordan kjemien uttrykkes i programkode.
2. **Koden blir lettere å tilpasse.** En liten funksjon med tydelige parametre kan endres dersom problemet får andre enheter, flere stofftyper eller nye betingelser.

Det er også lurt å vurdere kvaliteten på kjemisk notasjon og dokumentasjon i mindre biblioteker. Et program som skriver formler på en uvanlig måte, kan gjøre resultatene vanskeligere å lese.

### Støkiometri vi programmerer selv

Til stoffmengdeberegningene trenger vi den molare massen. Den kan hentes fra RDKit eller `mendeleev`; resten kan vi uttrykke med egne funksjoner.


In [ ]:
from rdkit import Chem
from rdkit.Chem import Descriptors


def molar_masse(smiles):
    """Molar masse i g/mol for et molekyl gitt ved SMILES."""
    molekyl = Chem.MolFromSmiles(smiles)
    if molekyl is None:
        raise ValueError("Ugyldig SMILES: " + smiles)
    return Descriptors.MolWt(molekyl)


def stoffmengde(masse, smiles):
    """Stoffmengde i mol for en gitt masse i gram."""
    return masse / molar_masse(smiles)


def masse(stoffmengde_mol, smiles):
    """Masse i gram for en gitt stoffmengde i mol."""
    return stoffmengde_mol * molar_masse(smiles)


butan_1_ol = "CCCCO"

print("Molar masse:", round(molar_masse(butan_1_ol), 2), "g/mol")
print("2.00 g tilsvarer", round(stoffmengde(2.00, butan_1_ol), 5), "mol")
print("0.150 mol veier", round(masse(0.150, butan_1_ol), 3), "g")


```{admonition} Underveisoppgave: Bygg ut verktøykassen di
:class: tip

Skriv dine egne funksjoner for følgende, med docstring og fornuftige parameternavn:

1. `konsentrasjon(masse, smiles, volum_liter)` som gir molaritet.
2. `fortynn(c1, v1, v2)` som gir sluttkonsentrasjonen etter fortynning.
3. `antall_molekyler(masse, smiles)` som bruker Avogadros tall.
4. `masseprosent(smiles, grunnstoffsymbol)` som gir masseprosenten av ett grunnstoff i forbindelsen. Til denne trenger du å iterere over atomene i molekylet med `molekyl.GetAtoms()` og `atom.GetSymbol()`, og du må huske på hydrogenatomene som ikke er skrevet ut. Se på `Chem.AddHs`.

Test hver funksjon mot en beregning du gjør for hånd. Det er hele poenget med å skrive dem selv.
```

````{admonition} Løsningsforslag
:class: tip, dropdown

```{code-block} python
AVOGADRO = 6.02214076e23   # per mol, eksakt definert


def konsentrasjon(masse, smiles, volum_liter):
    """Molaritet i mol/L."""
    return stoffmengde(masse, smiles) / volum_liter


def fortynn(c1, v1, v2):
    """Sluttkonsentrasjon etter fortynning fra volum v1 til v2."""
    return c1 * v1 / v2


def antall_molekyler(masse, smiles):
    """Antall molekyler i en gitt masse."""
    return stoffmengde(masse, smiles) * AVOGADRO


def masseprosent(smiles, grunnstoffsymbol):
    """Masseprosent av ett grunnstoff i en forbindelse."""
    from mendeleev import element

    molekyl = Chem.AddHs(Chem.MolFromSmiles(smiles))

    total = 0.0
    valgt = 0.0
    for atom in molekyl.GetAtoms():
        symbol = atom.GetSymbol()
        m = element(symbol).atomic_weight
        total += m
        if symbol == grunnstoffsymbol:
            valgt += m

    return 100 * valgt / total


print(round(masseprosent("CCCCO", "C"), 2), "% karbon i butan-1-ol")
```

Kontroll for hånd: butan-1-ol er C4H10O med M = 74.12 g/mol. Karbon bidrar med 4 · 12.011 = 48.04 g/mol, altså 64.8 %.
````


### Balansering med lineær algebra

En reaksjonslikning er balansert når antallet atomer av hvert grunnstoff er det samme på begge sider. Dersom koeffisientene behandles som ukjente, får vi én lineær likning for hvert grunnstoff. Balansering kan derfor formuleres som et **homogent lineært likningssystem**.

Vi balanserer ufullstendig forbrenning av benzen, der produktene er CO og vann:

$$a \cdot \mathrm{C_6H_6} + b \cdot \mathrm{O_2} \longrightarrow c \cdot \mathrm{CO} + d \cdot \mathrm{H_2O}$$

Vi skriver én likning for hvert grunnstoff og flytter alle ledd til samme side:

- Karbon: $6a-c=0$
- Hydrogen: $6a-2d=0$
- Oksygen: $2b-c-d=0$


In [ ]:
import sympy as sp

#                a   b   c   d
A = sp.Matrix([[ 6,  0, -1,  0],    # C
               [ 6,  0,  0, -2],    # H
               [ 0,  2, -1, -1]])   # O

losning = A.nullspace()[0]

# Nullrommet gir en retning, ikke bestemte tall. Vi skalerer opp til
# minste sett med hele tall ved å gange med fellesnevneren.
nevnere = [sp.Rational(x).q for x in losning]
koeffisienter = losning * sp.ilcm(*nevnere)

print("a, b, c, d =", list(koeffisienter))


Svaret er $2\,\mathrm{C_6H_6} + 9\,\mathrm{O_2} \longrightarrow 12\,\mathrm{CO} + 6\,\mathrm{H_2O}$.

Vi kontrollerer at reaksjonen inneholder 12 karbonatomer, 12 hydrogenatomer og 18 oksygenatomer på hver side.

Den viktigste faglige delen av arbeidet er å sette opp matrisen riktig. `sympy` løser deretter den lineære algebraen. På denne måten kan vi kontrollere både den kjemiske modellen og det numeriske resultatet.


```{admonition} Underveisoppgave: Balanser tre til
:class: tip

Sett opp matrisen og balanser følgende med metoden ovenfor. Kontroller hvert svar for hånd.

1. Fullstendig forbrenning av etanol: $\mathrm{C_2H_5OH} + \mathrm{O_2} \rightarrow \mathrm{CO_2} + \mathrm{H_2O}$
2. Framstilling av ammoniakk: $\mathrm{N_2} + \mathrm{H_2} \rightarrow \mathrm{NH_3}$
3. En redoksreaksjon i sur løsning: $\mathrm{MnO_4^-} + \mathrm{Fe^{2+}} + \mathrm{H^+} \rightarrow \mathrm{Mn^{2+}} + \mathrm{Fe^{3+}} + \mathrm{H_2O}$

Den siste krever noe mer: du må ta med **ladning** som en ekstra rad i matrisen, på samme måte som et grunnstoff. Hvorfor fungerer det?
```

````{admonition} Løsningsforslag
:class: tip, dropdown

Etanol, med rekkefølgen (C2H5OH, O2, CO2, H2O):

```{code-block} python
A = sp.Matrix([[ 2,  0, -1,  0],    # C
               [ 6,  0,  0, -2],    # H
               [ 1,  2, -2, -1]])   # O
```

Dette gir 1, 3, 2, 3.

For redoksreaksjonen med rekkefølgen (MnO4-, Fe2+, H+, Mn2+, Fe3+, H2O):

```{code-block} python
A = sp.Matrix([[ 1,  0,  0, -1,  0,  0],   # Mn
               [ 4,  0,  0,  0,  0, -1],   # O
               [ 0,  1,  0,  0, -1,  0],   # Fe
               [ 0,  0,  1,  0,  0, -2],   # H
               [-1,  2,  1, -2, -3,  0]])  # ladning
```

Svaret er 1, 5, 8, 1, 5, 4.

Ladning fungerer som en ekstra rad fordi ladningsbevaring er en bevaringslov av nøyaktig samme *matematiske* form som massebevaring: summen av ladning på venstre side må være lik summen på høyre. Matrisa bryr seg ikke om hva raden betyr fysisk, bare at det er en størrelse som skal balansere.
````


## Numeriske beregninger med bibliotek

Noen beregninger blir raskt omfattende, selv om vi kjenner de kjemiske prinsippene. Da kan et bibliotek være nyttig, særlig når det løser flere koblede likninger eller utfører mange beregninger etter hverandre.

### pHcalc

pH i en sterk syre kan ofte beregnes direkte. I en løsning med flere svake syrer, baser og ioner må derimot alle protolyselikevektene og ladningsbalansen behandles samtidig. Biblioteket `pHcalc` er laget for slike systemer.

Det bruker tre sentrale klasser:

- `Acid` for en syre med én eller flere $K_a$-verdier
- `Inert` for ioner som ikke deltar i protolyse, for eksempel Na⁺ eller Cl⁻
- `System` for løsningen som helhet

H₃O⁺ og OH⁻ legges ikke inn som egne komponenter. Konsentrasjonen av H₃O⁺ justeres til ladningsbalansen er oppfylt, mens OH⁻ følger av vannets ionprodukt.


In [ ]:
from pHcalc import Acid, Inert, System

# 0.010 M eddiksyre. Ka = 1.75e-5, og HA har ladning 0.
eddiksyre = Acid(Ka=1.75e-5, charge=0, conc=0.010)

losning = System(eddiksyre)
losning.pHsolve()

print("pH =", round(losning.pH, 3))


Her ser vi koblingen til kapitlet om numeriske metoder. `pHcalc` finner en løsning ved hjelp av en numerisk nullpunktsmetode. Vi kan formulere det samme problemet selv og kontrollere at metodene gir samme resultat.

For en enprotisk syre er ladningsbalansen

$$[\mathrm{H_3O^+}] = [\mathrm{A^-}] + [\mathrm{OH^-}]$$

Når vi setter inn $[\mathrm{A^-}] = \frac{K_a c}{K_a + [\mathrm{H_3O^+}]}$ og $[\mathrm{OH^-}] = K_w/[\mathrm{H_3O^+}]$, får vi et nullpunktsproblem med $[\mathrm{H_3O^+}]$ som ukjent.


In [ ]:
import numpy as np


def ladningsbalanse(h, Ka, c, Kw=1.0e-14):
    """Skal være null når h er riktig H3O+-konsentrasjon."""
    A_minus = Ka * c / (Ka + h)
    OH = Kw / h
    return h - A_minus - OH


def halveringsmetoden(f, a, b, tol=1e-18, n=200):
    """Finner nullpunktet til f i intervallet [a, b]."""
    for _ in range(n):
        c = (a + b) / 2
        if abs(f(c)) < tol:
            return c
        if f(a) * f(c) < 0:
            b = c
        else:
            a = c
    return (a + b) / 2


h = halveringsmetoden(lambda x: ladningsbalanse(x, 1.75e-5, 0.010),
                      1e-14, 1.0)

print("pH med egen kode:", round(-np.log10(h), 3))
print("pH med pHcalc:   ", round(losning.pH, 3))


Når vi først har formulert og kontrollert den enkle modellen selv, kan biblioteket brukes med større trygghet.

### Sammensatte systemer

For en treprotisk syre blir det flere koblede likevekter. Her er det hensiktsmessig å la biblioteket sette opp og løse systemet.


In [ ]:
# Fosforsyre, treprotisk. Ka-verdiene gis som ei liste.
fosforsyre = Acid(Ka=[7.5e-3, 6.2e-8, 4.8e-13], charge=0, conc=0.010)

losning = System(fosforsyre)
losning.pHsolve()
print("0.010 M H3PO4:  pH =", round(losning.pH, 3))

# Natriumdihydrogenfosfat: samme syre, men delvis nøytralisert.
# Ett Na+ per formelenhet.
natrium = Inert(charge=1, conc=0.010)
losning2 = System(fosforsyre, natrium)
losning2.pHsolve()
print("0.010 M NaH2PO4: pH =", round(losning2.pH, 3))


### Titrerkurver

En titrerkurve kan lages ved å beregne pH for en serie mengder tilsatt base. Med `pHcalc` kan dette gjøres i en løkke.


In [ ]:
import matplotlib.pyplot as plt

c_syre = 0.010
na_konsentrasjoner = np.linspace(1e-8, 0.020, 300)

pH_verdier = []
for c_na in na_konsentrasjoner:
    syre = Acid(Ka=1.75e-5, charge=0, conc=c_syre)
    base = Inert(charge=1, conc=c_na)
    system = System(syre, base)
    system.pHsolve()
    pH_verdier.append(system.pH)

plt.figure(figsize=(7, 4))
plt.plot(na_konsentrasjoner / c_syre, pH_verdier, color="teal")
plt.axvline(1.0, color="grey", linestyle="--", label="Ekvivalenspunkt")
plt.xlabel("Mol NaOH per mol syre")
plt.ylabel("pH")
plt.title("Titrering av 0.010 M eddiksyre med NaOH")
plt.legend()
plt.grid(alpha=0.3)
plt.show()


```{admonition} Underveisoppgave: Les kurven
:class: tip

1. Hvor på kurven ligger bufferområdet? Hva er pH i midten av det, og hva er sammenhengen med pKa til eddiksyre?
2. Hvorfor er pH ved ekvivalenspunktet *over* 7, og ikke lik 7?
3. Lag samme kurve for saltsyre (bruk `Inert(charge=-1, conc=0.010)` for Cl⁻). Hvordan skiller den seg fra kurven for eddiksyre, og hvorfor?
4. Lag kurven for fosforsyre. Hvor mange sprang ser du, og hvorfor ser du ikke like mange sprang som syren har protoner?
```

````{admonition} Løsningsforslag
:class: tip, dropdown

1. Bufferområdet er det flate partiet rundt halv nøytralisering, altså rundt 0.5 på x-aksen. Der er pH lik pKa, som for eddiksyre er omtrent 4.76.

2. Ved ekvivalenspunktet er all syren omdannet til acetat, som er den korresponderende basen til en svak syre. Acetat reagerer med vann og gir OH⁻, så løsningen blir basisk.

3. For saltsyre er det ingen buffersone. Kurven stiger nesten rett gjennom hele området og ekvivalenspunktet ligger på pH 7, fordi Cl⁻ ikke er en base av betydning.

4. Fosforsyre gir to tydelige sprang, ikke tre. Det tredje pKa-trinnet ligger på omtrent 12.3, og der er løsningen så basisk at vannets egen protolyse dominerer og jevner ut spranget.
````


## Vurdere et bibliotek

Det viktigste i dette kapitlet er ikke å huske syntaksen til hvert bibliotek, men å kunne vurdere om et verktøy passer til oppgaven. Følgende spørsmål er nyttige:

**1. Er biblioteket vedlikeholdt og dokumentert?** Se etter oppdatert dokumentasjon, nyere versjoner og aktivitet i prosjektet. Et eldre bibliotek kan fortsatt fungere godt, men bør testes med Python-versjonen og de andre pakkene du bruker.

**2. Hva tilfører biblioteket?** Tilgang til data, omfattende algoritmer og standardiserte formater er gode grunner til å bruke et bibliotek. For svært enkle beregninger kan en egen funksjon være tydeligere.

**3. Hvilke avhengigheter og begrensninger har det?** Undersøk hvilke andre pakker som må installeres, hvilke filformater som støttes, og om biblioteket fungerer i miljøet du skal bruke.

**4. Kan resultatet kontrolleres?** Test biblioteket på et tilfelle der du kjenner svaret fra en håndberegning, en tabellverdi eller en annen pålitelig kilde.

```{admonition} Dette gjelder også KI-generert kode
:class: important

KI-verktøy kan foreslå biblioteker og funksjoner som ser plausible ut, men som ikke finnes eller brukes på en annen måte enn foreslått. Kontroller derfor både dokumentasjonen og resultatet før koden tas i bruk.
```


```{admonition} Underveisoppgave: Vurder et ukjent bibliotek
:class: tip

Søk opp et Python-bibliotek for kjemi som ikke er nevnt i dette kapitlet. Forslag: `chempy`, `periodictable`, `molmass`, `pymatgen`, `ase` eller `cclib`.

Skriv en kort vurdering på fem til ti setninger:

1. Når kom siste versjon?
2. Hvilken av de fire kategoriene i dette kapitlet hører det hjemme i?
3. Hva gir det deg som du ikke kan skrive selv?
4. Finn ett eksempel i dokumentasjonen, kjør det, og kontroller svaret mot noe du kan regne ut eller slå opp.
5. Ville du brukt det? Begrunn.
```


## Sluttoppgaver

Disse oppgavene kombinerer flere av bibliotekene og krever at du bruker det du har lært om datahåndtering tidligere i emnet.

```{admonition} Oppgave 1: Trender i periodesystemet
:class: tip

Bruk `fetch_table` til å hente hele periodesystemet.

1. Lag et plott av kovalent radius mot atomnummer for grunnstoff 1 til 86. Fargelegg punktene etter blokk (s, p, d, f).
2. Marker starten på hver nye periode med en loddrett strek.
3. Beskriv sagtannmønsteret du ser og forklar det kjemisk.
4. Lag et andre plott av første ioniseringsenergi mot atomnummer i samme figur, med egen y-akse. Hva er sammenhengen mellom de to kurvene, og hvorfor?
5. Finn de tre grunnstoffene som avviker mest fra den generelle trenden i ioniseringsenergi innenfor periode 2. Forklar hvert avvik.
```

```{admonition} Oppgave 2: Fra navn til struktur til egenskap
:class: tip

Velg ti legemidler eller naturstoffer.

1. Hent CID, molekylformel og molar masse fra PubChem.
2. Hent SMILES for hver av dem (fra PubChem eller ved å slå det opp) og les dem inn i RDKit.
3. Kontroller at molar masse fra PubChem og fra RDKit stemmer overens. Hvis de avviker, finn ut hvorfor. (Hint: se på hva som skjer med salter og hydrater.)
4. Bygg en dataframe med minst fem RDKit-deskriptorer.
5. Lag et spredningsplott av logP mot TPSA og merk punktene med navn. Ser du noe mønster?
6. Tegn alle ti strukturene i et rutenett med `Draw.MolsToGridImage`.
```

```{admonition} Oppgave 3: Titrering med to metoder
:class: tip

Du skal titrere 25.0 mL 0.100 M maursyre (Ka = 1.8 · 10⁻⁴) med 0.100 M NaOH.

1. Beregn pH før tilsetting, ved halv nøytralisering, ved ekvivalenspunktet og etter 5 mL overskudd, ved hjelp av **din egen** nullpunktsalgoritme fra kapitlet om numeriske metoder.
2. Beregn de samme fire punktene med `pHcalc`.
3. Lag hele titrerkurven med `pHcalc` og marker de fire punktene på den.
4. Sammenlikn de to metodene. Hvor stort er avviket, og hva skyldes det?
5. Diskuter kort: i hvilke situasjoner ville du brukt din egen kode, og i hvilke ville du brukt biblioteket?
```

```{admonition} Oppgave 4: Balansering og utbytte
:class: tip

Termitt-reaksjonen er $\mathrm{Fe_2O_3} + \mathrm{Al} \rightarrow \mathrm{Fe} + \mathrm{Al_2O_3}$.

1. Balanser den med matrisemetoden fra del 3.
2. Skriv en funksjon som tar balanserte koeffisienter, molare masser og utgangsmasser, og finner ut hvilket stoff som er begrensende reaktant.
3. Beregn teoretisk utbytte av jern når du starter med 50.0 g Fe₂O₃ og 20.0 g Al.
4. Utvid funksjonen slik at den også håndterer prosentvis utbytte når du oppgir faktisk utbytte.
5. Test funksjonen din på minst to andre reaksjoner der du kjenner fasiten.
```

```{admonition} Oppgave 5: Din egen bibliotekvurdering
:class: tip

Finn et kjemibibliotek som du mener burde vært med i dette kapitlet, eller et som du mener bør brukes med forsiktighet.

Skriv en kort tekst på en halv til én side der du:

1. Beskriver hva biblioteket gjør, med minst ett kjørende kodeeksempel.
2. Plasserer det i en av de fire kategoriene.
3. Vurderer det etter de fire spørsmålene i avsnittet ovenfor.
4. Konkluderer med en anbefaling.

Ta med kontrollen du gjorde av svaret. Ta med kontrollen du har gjort, slik at vurderingen bygger på et etterprøvbart eksempel.
```
